In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pib_helper import load_psi4_molecule, calculate_r_squared
import psi4

psi4.set_output_file('output.dat', False)
psi4.set_memory('1 GB')

<a id="part4"></a>

## Part 4 — Comparing Models

We now have two models for the electronic structure of conjugated polyenes:
- The **particle-in-a-box (PIB)** — a simple analytic formula with no adjustable parameters.
- **Hartree–Fock (HF)** — a detailed quantum-chemical calculation.

How do they compare to each other, and to **experiment**? And can we improve a simple model by fitting it to match experimental data?

This is a common strategy in computational chemistry: use a cheap model with empirically fitted parameters to approximate expensive calculations or experiments. If the cheap model captures the right *shape* of the relationship, a linear correction can bring the absolute values into agreement:

$$E_{\text{corrected}} = \alpha \cdot E_{\text{model}} + \beta$$

### Polyenes for this study

| $n_c$ | Name | XYZ file |
|-------|------|----------|
| 4 | butadiene | `data/butadiene.xyz` |
| 6 | hexatriene | `data/hexatriene.xyz` |
| 8 | octatetraene | `data/octatetraene.xyz` |
| 10 | decapentaene | `data/decapentaene.xyz` |
| 12 | dodecahexaene | `data/dodecahexaene.xyz` |
| 14 | tetradecaheptaene | `data/tetradecaheptaene.xyz` |
| 16 | hexadecaoctaene | `data/hexadecaoctaene.xyz` |

### Key formulas

- Total electrons: $n_e = 7 n_c + 2$
- HOMO index (zero-indexed): $n_e / 2 - 1$
- Conversion: 1 Hartree = 27.211 eV

---

### Coding Activity 4
`20 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.
- P4.2: Fit nonlinear curves to data.

#### Part A — HF HOMO–LUMO Gaps

The cell below runs a Hartree–Fock calculation on **ethane** and extracts the HOMO–LUMO gap. Study it carefully — you will adapt this approach for the polyenes.

Ethane (C$_2$H$_6$) has $n_e = 18$ electrons, so the HOMO is at orbital index 8 (zero-indexed: $18/2 - 1 = 8$). The molecular geometry is loaded from a pre-built XYZ file using `load_psi4_molecule()`.

In [ ]:
# --- Worked example: ethane HOMO-LUMO gap ---
mol_ethane = load_psi4_molecule('data/ethane.xyz')
energy_ethane, wfn_ethane = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol_ethane)

eps_ethane = wfn_ethane.epsilon_a().np
n_e_ethane = 18
homo_idx = n_e_ethane // 2 - 1
lumo_idx = homo_idx + 1
gap_ethane = (eps_ethane[lumo_idx] - eps_ethane[homo_idx]) * 27.211  # convert to eV

print(f'Ethane HF/STO-3G energy: {energy_ethane:.6f} a.u.')
print(f'HOMO index: {homo_idx}, LUMO index: {lumo_idx}')
print(f'HOMO-LUMO gap: {gap_ethane:.2f} eV')

In [ ]:
# Subgoal: Compute HF HOMO-LUMO gaps for the polyenes

polyene_nc = [4, 6, 8, 10, 12, 14, 16]
polyene_xyz = [
    'data/butadiene.xyz',
    'data/hexatriene.xyz',
    'data/octatetraene.xyz',
    'data/decapentaene.xyz',
    'data/dodecahexaene.xyz',
    'data/tetradecaheptaene.xyz',
    'data/hexadecaoctaene.xyz',
]

hf_gaps = []

# YOUR CODE HERE
# Loop over polyene_nc and polyene_xyz
# For each molecule:
#   1. Load the psi4 molecule with load_psi4_molecule(xyz_file)
#   2. Run psi4.energy('SCF/STO-3G', ...)
#   3. Get orbital energies with wfn.epsilon_a().np
#   4. Compute n_e = 7 * n_c + 2, homo_idx = n_e // 2 - 1
#   5. Compute gap in eV and append to hf_gaps

print('HF HOMO-LUMO gaps (eV):', hf_gaps)

In [ ]:
# Subgoal: Plot HF gaps vs chain length

# YOUR CODE HERE


#### Part B — PIB HOMO–LUMO Gaps

The particle-in-a-box model gives the energy of level $n$ as:

$$E_n = \frac{n^2 \pi^2}{2 L^2} \quad \text{(atomic units)}$$

For a polyene with $n_c$ carbon atoms:
- Box length: $L = 1.4 \times (n_c - 1)$ Å, converted to a.u. by multiplying by 1.8897
- HOMO quantum number: $n_{\text{HOMO}} = n_c / 2$
- LUMO quantum number: $n_{\text{LUMO}} = n_c / 2 + 1$
- Gap: $\Delta E = E_{\text{LUMO}} - E_{\text{HOMO}}$, converted to eV by multiplying by 27.211

In [ ]:
# Subgoal: Write a function that returns the PIB HOMO-LUMO gap in eV

def pib_energy_gap(n_c):
    """
    Compute the PIB HOMO-LUMO gap for a polyene with n_c carbons.
    
    Parameters
    ----------
    n_c : int
        Number of carbon atoms.
    
    Returns
    -------
    gap_eV : float
        HOMO-LUMO gap in eV.
    """
    # YOUR CODE HERE
    # 1. Compute L in Angstroms: L = 1.4 * (n_c - 1)
    # 2. Convert to atomic units: L_au = L * 1.8897
    # 3. n_homo = n_c // 2, n_lumo = n_c // 2 + 1
    # 4. E_n = n^2 * pi^2 / (2 * L_au^2)
    # 5. gap = (E_lumo - E_homo) * 27.211


In [ ]:
# Subgoal: Compute PIB gaps and plot alongside HF gaps

pib_gaps = []

# YOUR CODE HERE
# 1. Loop over polyene_nc and compute pib_energy_gap for each
# 2. Plot both hf_gaps and pib_gaps vs polyene_nc on the same axes
#    with labels, legend, axis labels, and title


#### Part C — Comparison to Experiment

We have experimental UV absorption data for these polyenes. Let's see how both models compare to reality.

We'll also apply a **linear correction** to each model:

$$E_{\text{corrected}} = \alpha \cdot E_{\text{model}} + \beta$$

This fits two parameters ($\alpha$, $\beta$) to best match the experimental data. If the model captures the right *trend*, the correction should give good agreement.

In [ ]:
# --- Given: load experimental data ---
expt_data = pd.read_csv('data/polyene_excitation.csv')
print(expt_data)

expt_energies = expt_data['excitation_energy_eV'].values

In [ ]:
# Subgoal: Plot all three datasets (HF, PIB, experiment) on the same axes

# YOUR CODE HERE
# Plot hf_gaps, pib_gaps, and expt_energies vs polyene_nc
# Use different markers/colors and a legend


In [ ]:
# Subgoal: Fit linear corrections to both models

def linear_model(x, alpha, beta):
    """Linear correction: alpha * x + beta."""
    return alpha * x + beta

# YOUR CODE HERE
# 1. Use curve_fit to fit linear_model to (hf_gaps, expt_energies)
#    Store result as popt_hf
# 2. Use curve_fit to fit linear_model to (pib_gaps, expt_energies)
#    Store result as popt_pib
# 3. Print the fitted parameters for each model


In [ ]:
# Subgoal: Compare corrected predictions to experiment

# YOUR CODE HERE
# 1. Compute corrected predictions:
#    hf_corrected = linear_model(np.array(hf_gaps), *popt_hf)
#    pib_corrected = linear_model(np.array(pib_gaps), *popt_pib)
# 2. Plot expt_energies, hf_corrected, pib_corrected vs polyene_nc
# 3. Compute and print R^2 for each:
#    calculate_r_squared(expt_energies, hf_corrected)
#    calculate_r_squared(expt_energies, pib_corrected)


### Question 4
`10 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.

a) Before correction: which model (HF or PIB) is closer to the experimental values? Is either one quantitatively accurate?

b) After the linear correction: compare the $R^2$ values. What does this tell you about how well each model captures the *trend* in excitation energies?

c) What are the advantages of using a cheap fitted model (like corrected PIB) compared to running a full HF calculation every time?

---

*Your answer here (`double click me!`):*

---

<a id="reflection"></a>

## Reflection
`10 points`

a) In this lab, you worked with models at several levels of detail: a simple analytic formula (PIB), a numerical variational procedure, and a full quantum-chemical method (Hartree–Fock). In your own words, what makes a model "useful" even when it is known to be imperfect?

b) Describe one situation in this lab where a numerical procedure (optimization, curve fitting, or iteration) was essential — where you could not have obtained the result by hand calculation alone. What did you learn from this?

c) What is one thing from this lab that you found surprising or that changed how you think about quantum mechanics or computational chemistry?

*Your answer here (`double click me!`):*